# 目的
* JalcのメタデータをApache Solrに登録

## <font color=red>要実行</font>

In [1]:
# モジュール及び接続文字列の読み込み
import pymongo
import re
from itertools import permutations
import pysolr
url_mongodb = "mongodb://localhost:27017/"
solr_url = "http://localhost:8983/solr/jalc"

# MongoDBへの接続
myclient = pymongo.MongoClient(url_mongodb)
mydb = myclient["jalc"]
mycol = mydb["restapi"]

# solrへの接続
solr = pysolr.Solr(solr_url,always_commit=True)

## 登録処理
* Apache Solr のコア作成・スキーマ定義が完了している前提

In [2]:
# ドキュメントが全フィールドを持つか
def has_all_fields(doc):
    return bool(
        ("title_list" in doc) 
        and ("creator_list" in doc) 
        and ("publication_date" in doc) 
        and ("journal_title_name_list" in doc) 
        and (("volume" in doc) or ("issue" in doc)) 
        and (("first_page" in doc) or ("last_page" in doc))
    )

# 著者名の取得
def get_creator(doc):
    regex = re.compile(r'[^ぁ-んァ-ン一-龯]') # 日本語以外の範囲
    creator_list = []
    for creator in doc.get("creator_list"):
        for name in creator.get("names"):
            last_name = name.get("last_name","")
            creator_list += last_name.split()
            first_name = name.get("first_name","")
            creator_list += first_name.split()
            # 日本語の場合、姓名を繋げた文字列を追加
            if not (regex.search(first_name) and regex.search(last_name)):
                creator_list.append(last_name + first_name)
            # 英語の場合、イニシャルの組み合わせを追加
            if regex.search(first_name):
                split_name = re.split(r"[,\.\s\-\(\)]", first_name)
                names = [n for n in split_name if n]
                creator_list += names
                # recallの計算用に全てのイニシャルの順列の組み合わせ（2個まで）を用意しておく
                initials = [e[0].upper() for e in names if re.search(r"[a-zA-Z]", e[0])]
                for r in range(1, 3):
                    for perm in permutations(initials, r):
                        creator_list.append("".join(perm))
    creator_list = list(set(creator_list))              
    return creator_list         

# 筆頭著者の取得
def get_first_author(doc):
    regex = re.compile(r'[^ぁ-んァ-ン一-龯]')
    fa_names = []
    fa = doc.get("creator_list")[0]
    for name in fa.get("names"):
        last_name = name.get("last_name", "")
        first_name = name.get("first_name", "")
        # 姓があればそれを追加、なければ名を追加
        if last_name:
            fa_names.append(last_name)
        else:
            fa_names.append(first_name)
        # 日本語の場合は姓と名をつなげた文字列も追加（姓が1文字だと2gramで拾えないため）
        if not (regex.search(first_name) and regex.search(last_name)):
            fa_names.append(last_name + " " + first_name)
    fa_names = list(set(fa_names))
    return fa_names

# タイトルの取得
def get_title(doc):
    title_list = []
    for t in doc.get("title_list"):
        title_list.append(t.get("title"))
        if (subtitle := t.get("subtitle")) is not None:
            title_list.append(t.get("title") + " | " + subtitle)
    title_list = list(set(title_list))
    return title_list

# 雑誌名の取得
def get_journal(doc):
    journal_list = []
    for journal in doc.get("journal_title_name_list"):
        journal_list.append(journal.get("journal_title_name"))
    journal_list = list(set(journal_list))
    return journal_list    

# 出版年の取得
def get_issued(doc):
    issued = list(doc.get("publication_date").values())
    issued = [e[:4] for e in issued]
    return issued

# 巻号の取得
def get_volume_and_issue(doc):
    volume_and_issue = []
    volume_and_issue.append(doc.get("volume"))
    volume_and_issue.append(doc.get("issue"))
    volume_and_issue = list(set(volume_and_issue))  # setにする必要がないかも
    return volume_and_issue

# ページの取得
def get_page_range(doc):
    page_range = []
    page_range.append(doc.get("first_page"))
    page_range.append(doc.get("last_page"))
    page_range = "-".join([e for e in page_range if e])
    return [page_range]

# 出版者の取得
def get_publisher(doc):
    publisher_list = []
    if doc.get("publisher_list","") == "":
        return publisher_list
    for publisher in doc.get("publisher_list"):
        if publisher == None:
            publisher_list.append("")
        else:    
            publisher_list.append(publisher.get("publisher_name"))
    publisher_list = list(set(publisher_list))
    return publisher_list
    
# 以下本処理
documents = mycol.find({"content_type":"JA"})
count = 0
batch_size = 10000
doc_data_list = []

for doc in documents:
    if has_all_fields(doc):
        # 各フィールド情報の取得
        creator_list = get_creator(doc)
        # first_name,last_nameを持たないものがあった
        if creator_list == []:
            continue
        first_author = get_first_author(doc)
        title_list = get_title(doc)
        journal_list = get_journal(doc)
        issued = get_issued(doc)
        volume_and_issue = get_volume_and_issue(doc)
        page_range = get_page_range(doc)
        doi = doc.get("doi")
        publisher_list = get_publisher(doc)
        # 辞書型データに格納
        doc_data = {
            "doi":doi,
            "first_author":first_author,
            "creator":creator_list,
            "title":title_list,
            "journal":journal_list,
            "issued":issued,
            "volume_issue":volume_and_issue,
            "page_range":page_range,
            "publisher":publisher_list
        }
        doc_data_list.append(doc_data)
        count += 1
        # solrへ登録
        if len(doc_data_list) == batch_size:
            try:
                solr.add(doc_data_list)
                print(f"{len(doc_data_list)}件のデータを登録")
            except Exception as e:
                print("エラー発生")
                print(e)
            doc_data_list = []     
    else:
        continue    

# 残りのデータを登録
if doc_data_list:
    try:
        solr.add(doc_data_list)
        print(f"{len(doc_data_list)}件のデータを登録")
    except Exception as e:
        print("エラー発生")
        print(e)

print("全件登録完了")
print(count)

10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件のデータを登録
10000件

コア内の全データを削除

In [3]:
try:
    solr.delete(q='*:*')
    print("全件削除完了")
except Exception as e:
    print("エラー発生")
    print(e)    

全件削除完了


## schema API のコマンド
* ターミナルの方で実行

フィールドの定義
なお、jalcdataはコピーフィールドである

コピーフィールドルールの設定